In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


# **AfriSenti Error Analysis on the Twi Language**
## 1. Setup



In [4]:
# Imports
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2. Configuration and Data Loading


In [5]:
# Configuration
import warnings
warnings.filterwarnings('ignore')

# Paths (UPDATE WITH YOUR DRIVE PATH)
DRIVE_BASE = "/content/drive/MyDrive/AfriSenti/data"
LANG = "twi"  # Test language: twi, hau, yor, etc.

print(f"Loading data for {LANG}...")

# Load from Google Drive (already downloaded)
dataset = load_from_disk(f"{DRIVE_BASE}/{LANG}")

# Display dataset sizes
print(f"Train: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test: {len(dataset['test'])}")

# Display a raw example
print("\nRaw data example:")
print(f"Text: {dataset['train'][0]['tweet']}")
print(f"Label: {dataset['train'][0]['label']}")

Loading data for twi...
Train: 3481
Validation: 388
Test: 949

Raw data example:
Text: kako be shark but wo ti ewu
Label: 2


## 3. Baseline (Seed 2)



In [6]:
# 1. Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-base")
model = AutoModelForSequenceClassification.from_pretrained("Davlan/afro-xlmr-base", num_labels=3)

# 2. Tokenize
def tok(x):
    return tokenizer(x["tweet"], padding="max_length", truncation=True, max_length=200)

dataset = dataset.map(tok, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 3. Train
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        "./out", num_train_epochs=10, per_device_train_batch_size=32,
        learning_rate=1e-5, seed=2, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True, report_to="none",
    ),
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=lambda x: {"accuracy": accuracy_score(np.argmax(x.predictions, 1), x.label_ids)},
)

trainer.train()

# 4. Detailed evaluation
preds = trainer.predict(dataset["test"])
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print("\nPer-class metrics:")
print(classification_report(y_true, y_pred, target_names=["pos", "neu", "neg"]))

# 5. Save model
model.save_pretrained("/content/drive/MyDrive/AfriSenti/models/twi_baseline_seed2")
tokenizer.save_pretrained("/content/drive/MyDrive/AfriSenti/models/twi_baseline_seed2")
print("\n Model saved to Drive")

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


TEST SET RESULTS
Accuracy: 64.59%

Per-class metrics:
              precision    recall  f1-score   support

         pos       0.64      0.83      0.72       450
         neu       0.67      0.01      0.03       146
         neg       0.65      0.68      0.66       353

    accuracy                           0.65       949
   macro avg       0.65      0.51      0.47       949
weighted avg       0.65      0.65      0.59       949



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Model saved to Drive


## 4. Error Analysis

In [13]:
# Run the error analysis script
!python /content/drive/MyDrive/AfriSenti/error_analysis.py

Loading weights: 100% 201/201 [00:00<00:00, 869.49it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
Model loaded from Drive
Map: 100% 3481/3481 [00:00<00:00, 6321.12 examples/s]
Map: 100% 388/388 [00:00<00:00, 4989.36 examples/s]
Map: 100% 949/949 [00:00<00:00, 4246.25 examples/s]
Train: 3481 | Val: 388 | Test: 949
100% 49/49 [00:04<00:00, 12.15it/s]

Errors: 135/388 (34.8%)

CLASSIFICATION REPORT - VALIDATION SET
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

## **5. Interpretation of Results**

### Overall Performance
The model achieves **64.6% accuracy** on the test set. This is a reasonable baseline for a low-resource African language (random baseline would be 33%).

### Critical Issue: Neutral Class is Invisible
The model finds almost no neutral tweets. On the validation set, neutral recall is 0%. On the test set, it is only 1%. The neutral class is practically ignored.

**Why?** The neutral class is underrepresented (only 15% of the test set) and its examples are often ambiguous.

### Error Patterns
Among 135 validation errors:

- **Negative misclassified as positive**: 44 times
- **Neutral misclassified as positive**: 39 times
- **Positive misclassified as negative**: 33 times
- **Neutral misclassified as negative**: 19 times

**Key insight**: Neutral tweets are absorbed by both positive and negative classes. The model never keeps them as "neutral".

### What Causes the Errors?
- **Emojis** appear in 43% of errors. Smiling emojis (😂, 😁) dominate the model's prediction, pushing it toward "positive" even when the text is negative.
- **Repeated characters** appear in 31% of errors. Emphasis markers like "ooooo" and "paaa" are ignored by the model.
- **Exclamation and question marks** cause 0% of errors. These patterns work well and should be kept.

### Example of a Typical Error
> *"5mins biaa na scam nam mu 😁"* (True: negative → Predicted: positive)

The text contains "scam" (negative), but the 😁 emoji overrides the model's prediction.

### Conclusion
The model's main limitation is **data-related, not architectural**. The neutral class is underrepresented and emojis often dominate text sentiment.